In [0]:
%pip install pyyaml openpyxl requests

In [0]:
dbutils.library.restartPython()

In [0]:
import sys

from pathlib import Path
from datetime import date
from pyspark.sql import functions as F

CODE_ROOT = Path(
    "/Workspace/Users/arfaashaikh101@gmail.com/retail_dev_dbx"
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from scripts.bronze_loader import (
    load_config,
    prepare_source,
    table,
)
from scripts.transformations import create_layer_tables
from scripts.control import setup_control
from scripts.layer_runner import run_bronze

config = load_config(
    spark,
    CODE_ROOT / "config/source_contracts.yaml",
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

create_layer_tables(
    spark,
    config,
    {
        "bronze": "bronze_transactions",
        "monitoring": "retail_monitoring",
        "quarantine": "retail_quarantine",
    },
)

control_config = setup_control(
    spark,
    config["namespace"],
    CODE_ROOT / "config/control_contracts.yaml",
)

TRIGGER_TYPE = "MANUAL"
LOAD_ALL_HISTORY = False

In [0]:
from scripts.bronze_loader import prepare_source

source_path, source_manifest = prepare_source(config)

print("Source file:", source_path)
print("Source rows:", source_manifest["row_count"])

assert source_manifest["row_count"] == 1067371

In [0]:
def next_month(day):
    return (
        date(day.year + 1, 1, 1)
        if day.month == 12
        else date(day.year, day.month + 1, 1)
    )


cursor = date.fromisoformat(config["source"]["start_date"])
end = date.fromisoformat(
    config["source"]["historical_end_exclusive"]
)

batches = []

while cursor < end:
    batch_end = min(next_month(cursor), end)

    batches.append({
        "start": cursor.isoformat(),
        "end": batch_end.isoformat(),
    })

    cursor = batch_end

selected = batches if LOAD_ALL_HISTORY else batches[:1]

print("Selected batches:", len(selected))

In [0]:
results = []

for batch in selected:
    result = run_bronze(
        spark=spark,
        config=config,
        control_config=control_config,
        source_path=source_path,
        source_manifest=source_manifest,
        start=batch["start"],
        end=batch["end"],
        trigger_type=TRIGGER_TYPE,
        load_type="HISTORICAL",
    )

    results.append(result)
    print(result)

In [0]:
display(
    spark.table(control_config["tables"]["master"])
    .filter(F.col("layer") == "BRONZE")
    .select(
        "control_id",
        "batch_id",
        "run_id",
        "final_status",
        "write_outcome",
        "read_count",
        "accepted_count",
        "rejected_count",
        "committed_count",
        "target_delta_version",
        "replay_of_control_id",
        "failure_reason",
    )
    .orderBy(F.col("started_at").desc())
)